In [1]:
from app.utils.paramSQL import get_all_MET, get_all_MAT, get_all_THI, get_all_param_by_MAT_MET
from app.utils import paramSQL

In [27]:
all_params = get_all_param_by_MAT_MET(MET='MIG', MAT='Al-Si-5')
all_params_dict_index = {}  # 本轮目标：{DIA: {index: {paramName: paramValue}}}  index与THI一一对应
for DIA, paramName, paramValue, ParamIndex in all_params:
    if DIA not in all_params_dict_index:
        all_params_dict_index[DIA] = {}
    if ParamIndex not in all_params_dict_index[DIA]:
        all_params_dict_index[DIA][ParamIndex] = {}
    all_params_dict_index[DIA][ParamIndex][paramName] = paramValue

all_params_dict = {}  # 本轮目标：{DIA: {THI: {paramName: paramValue}}}  index与THI一一对应
for DIA, dict_of_this_DIA in all_params_dict_index.items():
    all_params_dict[DIA] = {}
    for param_index, dict_of_this_DIA_index in dict_of_this_DIA.items():
        all_params_dict[DIA][dict_of_this_DIA_index['Guideline value for material']] = dict_of_this_DIA_index
        
processed_params_dict = {}# 本轮目标：{DIA: {paramName: [(THI, paramValue)]}}
for DIA, params_THI_dict in all_params_dict.items():
    new_list_dict = {}
    for THI, params_this_DIA_THI in params_THI_dict.items():
        if THI == '-100':
            continue
        for paramName, paramValue in params_this_DIA_THI.items():
            if paramName not in new_list_dict:
                new_list_dict[paramName] = []
            new_list_dict[paramName].append((float(THI), float(paramValue)))
    # 存储的-100占位参数，不放进最终结果
    if new_list_dict:
        processed_params_dict[DIA] = new_list_dict

In [28]:
processed_params_dict

{'0.8': {'Feeder creep speed': [(0.6, 0.5),
   (0.8, 0.81),
   (1.0, 1.05),
   (1.5, 1.5),
   (2.0, 2.1),
   (3.0, 2.91),
   (4.0, 3.6),
   (5.0, 4.2),
   (6.0, 5.25),
   (10.0, 6.6)],
  'ArcingCurrent': [(0.6, 320.0),
   (0.8, 320.0),
   (1.0, 320.0),
   (1.5, 320.0),
   (2.0, 320.0),
   (3.0, 320.0),
   (4.0, 320.0),
   (5.0, 320.0),
   (6.0, 320.0),
   (10.0, 320.0)],
  'ArcingTime': [(0.6, 4.0),
   (0.8, 4.0),
   (1.0, 4.0),
   (1.5, 5.0),
   (2.0, 6.0),
   (3.0, 8.0),
   (4.0, 10.0),
   (5.0, 12.0),
   (6.0, 15.0),
   (10.0, 20.0)],
  'PulseBaseCurrent': [(0.6, 10.0),
   (0.8, 16.0),
   (1.0, 18.0),
   (1.5, 28.0),
   (2.0, 40.0),
   (3.0, 60.0),
   (4.0, 90.0),
   (5.0, 110.0),
   (6.0, 150.0),
   (10.0, 183.0)],
  'Current-rise': [(0.6, 300.0),
   (0.8, 300.0),
   (1.0, 300.0),
   (1.5, 300.0),
   (2.0, 300.0),
   (3.0, 300.0),
   (4.0, 300.0),
   (5.0, 300.0),
   (6.0, 300.0),
   (10.0, 300.0)],
  'Current-rise(tau)': [(0.6, 0.2),
   (0.8, 0.2),
   (1.0, 0.2),
   (1.5, 0.2),
  

In [51]:
import numpy as np

def linear_fit_and_predict(data, x):
    # 分离二维元组列表中的 x 和 y 值
    x_values = np.array([point[0] for point in data])
    y_values = np.array([point[1] for point in data])

    # 进行线性拟合，得到斜率和截距
    slope, intercept = np.polyfit(x_values, y_values, 1)

    # 定义线性方程
    def linear_equation(x):
        return slope * x + intercept

    # 使用线性方程预测给定 x 对应的 y 值
    predicted_y = linear_equation(x)

    return predicted_y

In [36]:
linear_fit_and_predict(data=processed_params_dict['0.8']['ArcingCurrent'], x=0.6)

-8.156985169696046e-15 320.00000000000006


320.00000000000006

In [38]:
def linear_interpolation(data, x):
    # 对数据按照 x 值排序
    sorted_data = sorted(data, key=lambda point: point[0])
    # 找到最接近的两个点
    for i in range(len(sorted_data) - 1):
        if sorted_data[i][0] <= x <= sorted_data[i + 1][0]:
            x1, y1 = sorted_data[i]
            x2, y2 = sorted_data[i + 1]
            # 线性插值公式
            y = y1 + (y2 - y1) * (x - x1) / (x2 - x1)
            return y
    # 如果 x 不在已有值的区间里
    return linear_fit_and_predict(data, x)

In [41]:
linear_interpolation(data=processed_params_dict['0.8']['Feeder creep speed'], x=0.66)

0.5930000000000001

In [52]:
def thickness_recommend(MAT, MET, THI):
    all_params = get_all_param_by_MAT_MET(MET, MAT)
    all_params_dict_index = {}  # 本轮目标：{DIA: {index: {paramName: paramValue}}}  index与THI一一对应
    for DIA, paramName, paramValue, ParamIndex in all_params:
        if DIA not in all_params_dict_index:
            all_params_dict_index[DIA] = {}
        if ParamIndex not in all_params_dict_index[DIA]:
            all_params_dict_index[DIA][ParamIndex] = {}
        all_params_dict_index[DIA][ParamIndex][paramName] = paramValue

    all_params_dict = {}  # 本轮目标：{DIA: {THI: {paramName: paramValue}}}  index与THI一一对应
    for DIA, dict_of_this_DIA in all_params_dict_index.items():
        all_params_dict[DIA] = {}
        for param_index, dict_of_this_DIA_index in dict_of_this_DIA.items():
            all_params_dict[DIA][dict_of_this_DIA_index['Guideline value for material']] = dict_of_this_DIA_index
            
    processed_params_dict = {}# 本轮目标：{DIA: {paramName: [(THI, paramValue)]}}
    for DIA, params_THI_dict in all_params_dict.items():
        new_list_dict = {}
        for _THI, params_this_DIA_THI in params_THI_dict.items():
            if _THI == '-100':
                continue
            for paramName, paramValue in params_this_DIA_THI.items():
                if paramName not in new_list_dict:
                    new_list_dict[paramName] = []
                new_list_dict[paramName].append((float(_THI), float(paramValue)))
        # 存储的-100占位参数，不放进最终结果
        if new_list_dict:
            processed_params_dict[DIA] = new_list_dict

    # 对于已解析的结果，进行线性预测
    result_dict = {}
    for DIA, params_dict_of_this_DIA in processed_params_dict.items():
        result_dict[DIA] = {}
        for param_name, data in params_dict_of_this_DIA.items():
            predict_param_value = linear_interpolation(data=data, x=THI)
            result_dict[DIA][param_name] = round(predict_param_value, 2)
    return result_dict

In [60]:
thickness_recommend(MET='MIG', MAT='Al-Si-5', THI=0.98)

{'0.8': {'Feeder creep speed': 1.03,
  'ArcingCurrent': 320.0,
  'ArcingTime': 4.0,
  'PulseBaseCurrent': 17.8,
  'Current-rise': 300.0,
  'Current-rise(tau)': 0.2,
  'PulsingCurrent': 210.0,
  'PulseingTime': 1.0,
  'Current-drop': 300.0,
  'Current-drop(Tau)': 0.2,
  'Droplet-detachment current': 29.5,
  'Driplet-detachment time': 0.3,
  'PulseFreqency': 51.2,
  'WireFeedSpeed': 3.42,
  'Voltage command value': 19.29,
  'Fact-I_b-control(pi)': 0.9,
  'Fact-I_p1-control(pi)': 15.0,
  'Fact-f-control(p)': 9.5,
  'Fact-I-b-correction': 0.0,
  'Fact-I-p1-correction': 0.0,
  'Fact-f-correction': 0.0,
  'Current-rise(short circuit)': 60.0,
  'Burn-back time': 0.0,
  'Guideline value for material': 0.98,
  'Amperage guideline value': 30.3,
  'Voltage guideline value': 16.89},
 '1': {'Feeder creep speed': 0.59,
  'ArcingCurrent': 400.0,
  'ArcingTime': 4.0,
  'PulseBaseCurrent': 14.9,
  'Current-rise': 300.0,
  'Current-rise(tau)': 0.1,
  'PulsingCurrent': 271.0,
  'PulseingTime': 1.0,
  'Cu

In [46]:
THI

'-100'